# ⚡ End-to-End Solid-State Battery Degradation Analysis & Autonomous AI Agent

Welcome to this comprehensive machine learning and AI agent project. This notebook showcases an end-to-end pipeline to predict and analyze the degradation of solid-state lithium battery cells under various physical design and environmental conditions.

## 🔋 Project Overview
Solid-state batteries (SSBs) are the next frontier in energy storage, replacing liquid electrolytes with solid counterparts to offer higher energy density and improved safety. However, mechanical stresses, interface degradation, and contact loss under cycle life are major bottlenecks. 

This project implements:
1. **Physics-inspired Synthetic Data Generation**: Simulating solid-state battery degradation profiles based on electrolyte composition, thickness, temperature, operating C-rate, and stack pressure.
2. **Machine Learning Pipeline**: Training Random Forest Regressors to predict capacity retention and interfacial resistance growth.
3. **Autonomous ReAct Agent (OrbitAgent)**: An AI agent that writes Python scripts to query our ML models, conducts Wikipedia literature searches, and synthesizes comprehensive research briefs.

### 1. Setup and Libraries

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error
from sklearn.preprocessing import LabelEncoder
import joblib
import os

sns.set_theme(style="darkgrid")
print("Libraries loaded successfully!")

### 2. Physics-Inspired Data Simulation

We generate a synthetic dataset of solid-state battery cells. The features reflect physical parameters:
- `electrolyte_type`: Sulfide, Oxide, Polymer
- `electrolyte_thickness_um`: thickness in micrometers (20 - 150 um)
- `c_rate`: rate of charging/discharging (0.1C to 3C)
- `temperature_c`: operating temp (10°C to 60°C)
- `pressure_mpa`: stack pressure applied to maintain contact (0.1 MPa to 10 MPa)
- `cycle_number`: index of the cycle (1 to 2000)

Targets:
- `capacity_retention`: percentage of initial capacity
- `interfacial_resistance_ohm`: cell interface resistance growth

In [ ]:
from ml_pipeline import generate_battery_data

# Generate 3000 samples for exploration
df = generate_battery_data(num_samples=3000, seed=42)
df.head(10)

### 3. Exploratory Data Analysis (EDA)
Let's see how our variables correlate and behave.

In [ ]:
plt.figure(figsize=(10, 6))
sns.scatterplot(data=df, x='cycle_number', y='capacity_retention', hue='electrolyte_type', alpha=0.5)
plt.title('Capacity Retention decay over Cycle Number by Electrolyte Type')
plt.xlabel('Cycle Number')
plt.ylabel('Capacity Retention (%)')
plt.show()

In [ ]:
plt.figure(figsize=(10, 6))
sns.boxplot(data=df, x='electrolyte_type', y='interfacial_resistance_ohm')
plt.title('Distribution of Interfacial Resistance by Electrolyte Type')
plt.show()

### 4. Model Training & Diagnostics

In [ ]:
# Preprocess categories
le = LabelEncoder()
df_encoded = df.copy()
df_encoded['electrolyte_type'] = le.fit_transform(df['electrolyte_type'])

X = df_encoded[['electrolyte_type', 'electrolyte_thickness_um', 'c_rate', 'temperature_c', 'pressure_mpa', 'cycle_number']]
y_cap = df_encoded['capacity_retention']
y_res = df_encoded['interfacial_resistance_ohm']

X_train, X_test, y_cap_train, y_cap_test, y_res_train, y_res_test = train_test_split(
    X, y_cap, y_res, test_size=0.2, random_state=42
)

print(f"Training set size: {X_train.shape[0]} samples")
print(f"Test set size: {X_test.shape[0]} samples")

In [ ]:
# Train Random Forest Regressors
model_cap = RandomForestRegressor(n_estimators=100, max_depth=12, random_state=42, n_jobs=-1)
model_cap.fit(X_train, y_cap_train)

model_res = RandomForestRegressor(n_estimators=100, max_depth=12, random_state=42, n_jobs=-1)
model_res.fit(X_train, y_res_train)

# Evaluate Capacity
pred_cap = model_cap.predict(X_test)
print('--- Capacity Retention Model ---')
print(f'R2 Score:  {r2_score(y_cap_test, pred_cap):.4f}')
print(f'MSE:       {mean_squared_error(y_cap_test, pred_cap):.4f}')
print(f'MAE:       {mean_absolute_error(y_cap_test, pred_cap):.4f}')

# Evaluate Resistance
pred_res = model_res.predict(X_test)
print('\n--- Interfacial Resistance Model ---')
print(f'R2 Score:  {r2_score(y_res_test, pred_res):.4f}')
print(f'MSE:       {mean_squared_error(y_res_test, pred_res):.4f}')
print(f'MAE:       {mean_absolute_error(y_res_test, pred_res):.4f}')

### 5. Feature Importances

In [ ]:
features_list = X.columns.tolist()
importances_df = pd.DataFrame({
    'Feature': features_list,
    'Capacity Importance': model_cap.feature_importances_,
    'Resistance Importance': model_res.feature_importances_
}).sort_values(by='Capacity Importance', ascending=False)

plt.figure(figsize=(10, 5))
sns.barplot(data=importances_df, x='Capacity Importance', y='Feature', color='royalblue')
plt.title('Capacity Retention - Feature Importance')
plt.show()

### 6. Interactive OrbitAgent Loop

Now we import the `OrbitAgent` from `agent.py` to see the ReAct planning cycle in action.

In [ ]:
from agent import OrbitAgent

# Initialize agent
agent = OrbitAgent()

# Define user goal
goal = "Predict the degradation profile of Oxide electrolyte battery cell at 45 C under 3 MPa pressure after 1200 cycles."

# Run agent loop
print(f"Starting Orbit Agent reasoning for: '{goal}'\n")
report = agent.run_loop(goal)

print("\n--- AGENT FINAL REPORT ---")
print(report)

### 7. Generate Submission file

This creates the `submission.csv` file expected by Kaggle based on predicting on the test set.

In [ ]:
from ml_pipeline import run_ml_pipeline

# Generate submission file
run_ml_pipeline()

# Preview the generated submission
sub_preview = pd.read_csv('submission.csv')
sub_preview.head()